# Pre-tournament World Cup predictions: 2010–2022

For each World Cup, this notebook fits models using only matches completed before the tournament's first match. It then freezes predictions for every tournament fixture before reading actual outcomes for scoring. This reproduces the real forecasting sequence: **train → predict → reveal actuals → evaluate**.

Models: leakage-safe pre-match Elo probabilities; regularized multinomial logistic regression on pre-match Elo difference and neutral venue; separate regularized Poisson goal models; and a training-outcome-frequency reference.

In [1]:
from pathlib import Path
from math import exp, factorial
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.metrics import log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd(); YEARS=[2010,2014,2018,2022]; SEED=20260705
matches=pd.read_csv(ROOT/'data/processed/matches_with_elo.csv',parse_dates=['date']).sort_values(['date','match_id']).reset_index(drop=True)
labels={'H':0,'D':1,'A':2}; ordered=['H','D','A']

def x(frame): return np.column_stack([frame.elo_difference.to_numpy(float)/400,frame.neutral.astype(float)])
def poisson_outcomes(hr,ar,cap=10):
    hp=np.array([exp(-hr)*hr**k/factorial(k) for k in range(cap+1)]); ap=np.array([exp(-ar)*ar**k/factorial(k) for k in range(cap+1)]); m=np.outer(hp,ap); m/=m.sum()
    return np.array([np.tril(m,-1).sum(),np.trace(m),np.triu(m,1).sum()])
def brier(y,p):
    actual=np.eye(3)[y]; return float(np.mean(np.sum((p-actual)**2,axis=1)))

prediction_frames=[]; metric_rows=[]; boundaries=[]
for year in YEARS:
    tournament=matches[(matches.tournament=='FIFA World Cup') & (matches.date.dt.year==year)].copy()
    assert len(tournament)>=64, f'incomplete {year} tournament: {len(tournament)} matches'
    cutoff=tournament.date.min(); train=matches[(matches.date<cutoff) & (matches.date>='2000-01-01')].copy()
    assert train.date.max()<cutoff<=tournament.date.min() and not set(train.match_id)&set(tournament.match_id)
    boundaries.append({'world_cup':year,'training_rows':len(train),'train_end':train.date.max().date(),'prediction_cutoff':cutoff.date(),'tournament_matches':len(tournament)})

    # Generate and freeze probabilities without accessing tournament scores/results.
    logistic=make_pipeline(StandardScaler(),LogisticRegression(C=.1,max_iter=1000,random_state=SEED)).fit(x(train),train.result.map(labels))
    raw=logistic.predict_proba(x(tournament)); logistic_p=np.zeros_like(raw)
    for source,class_id in enumerate(logistic[-1].classes_): logistic_p[:,int(class_id)]=raw[:,source]
    home=PoissonRegressor(alpha=.1,max_iter=1000).fit(x(train),train.home_score); away=PoissonRegressor(alpha=.1,max_iter=1000).fit(x(train),train.away_score)
    hr,ar=home.predict(x(tournament)),away.predict(x(tournament)); poisson_p=np.vstack([poisson_outcomes(h,a) for h,a in zip(hr,ar)])
    elo_p=tournament[['elo_home_probability','elo_draw_probability','elo_away_probability']].to_numpy(float)
    frequency=np.tile(train.result.value_counts(normalize=True).reindex(ordered).to_numpy(),(len(tournament),1))
    frozen={'logistic':logistic_p.copy(),'poisson':poisson_p.copy(),'elo':elo_p.copy(),'frequency':frequency.copy()}
    for p in frozen.values(): assert np.isfinite(p).all() and (p>=0).all() and np.allclose(p.sum(1),1)

    # Reveal actuals only after every model prediction is frozen.
    actual=tournament.result.map(labels).to_numpy()
    for name,p in frozen.items(): metric_rows.append({'world_cup':year,'model':name,'matches':len(actual),'log_loss':log_loss(actual,p,labels=[0,1,2]),'brier':brier(actual,p),'accuracy':float((p.argmax(1)==actual).mean())})
    out=tournament[['match_id','date','home_team','away_team','home_score','away_score','result']].copy(); out.insert(0,'world_cup',year)
    for name,p in frozen.items():
        for i,label in enumerate(['home_win','draw','away_win']): out[f'{name}_{label}']=p[:,i]
        out[f'{name}_prediction']=np.array(ordered)[p.argmax(1)]
    out['poisson_expected_home_goals']=hr; out['poisson_expected_away_goals']=ar; prediction_frames.append(out)

predictions=pd.concat(prediction_frames,ignore_index=True); metrics=pd.DataFrame(metric_rows); boundaries=pd.DataFrame(boundaries)
aggregate=(metrics.groupby('model').apply(lambda g:pd.Series({'mean_log_loss':np.average(g.log_loss,weights=g.matches),'mean_brier':np.average(g.brier,weights=g.matches),'accuracy':np.average(g.accuracy,weights=g.matches)}),include_groups=False).sort_values('mean_log_loss'))
display(boundaries); display(metrics.pivot(index='world_cup',columns='model',values='log_loss').round(4)); display(aggregate.round(4))

assert predictions.groupby('world_cup').size().to_dict()=={y:64 for y in YEARS}
assert len(predictions)==256 and len(metrics)==16
assert predictions.filter(regex='_(home_win|draw|away_win)$').apply(np.isfinite).all().all()
assert (metrics[['log_loss','brier','accuracy']].apply(np.isfinite).all().all())
assert aggregate.index[0]!='frequency', 'trained models should beat the uninformed reference'

csv_path=ROOT/'reports/world_cup_predictions_2010_2022.csv'; predictions.to_csv(csv_path,index=False)
table=metrics.pivot(index='world_cup',columns='model',values='log_loss').round(4).to_csv()
summary=aggregate.round(4).to_csv()
report=f'''# World Cup Pre-Tournament Backtest (2010–2022)\n\nAll 64 predictions for each tournament were generated from matches strictly before its opening match, then frozen before comparison with actual results. Total: 256 matches.\n\n## Log loss by tournament\n\n```csv\n{table}```\n\n## Aggregate ranking\n\n```csv\n{summary}```\n\nLower log loss and Brier score are better. Accuracy is secondary because it ignores probability quality. The per-match audit file is `reports/world_cup_predictions_2010_2022.csv`. This is a historical backtest, not evidence that future performance is guaranteed.\n'''
(ROOT/'reports/world_cup_backtest_2010_2022.md').write_text(report,encoding='utf-8')
print('All 2010–2022 pre-tournament leakage, normalization, coverage, and metric checks passed.')

G:\Coding\nations-ai\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "G:\Coding\nations-ai\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


,world_cup,training_rows,train_end,prediction_cutoff,tournament_matches
0,2010,9802,2010-06-08,2010-06-11,64
1,2014,13799,2014-06-10,2014-06-12,64
2,2018,17577,2018-06-12,2018-06-14,64
3,2022,21638,2022-11-19,2022-11-20,64


model,elo,frequency,logistic,poisson
world_cup,,,,
2010,0.9837,1.1113,0.9720,0.9911
2014,0.9780,1.0596,0.9692,0.9779
2018,0.9709,1.0923,0.9670,0.9848
2022,1.0311,1.0729,1.0372,1.0168


,mean_log_loss,mean_brier,accuracy
model,,,
logistic,0.9864,0.5818,0.5547
elo,0.9909,0.5884,0.5352
poisson,0.9927,0.5897,0.5508
frequency,1.0840,0.6588,0.4141


All 2010–2022 pre-tournament leakage, normalization, coverage, and metric checks passed.
